# Direct TS Prediction

Instead of searching along a path, we generate TS candidates directly using an AI model.

## How does it work?

tsdiff uses a diffusion model (hence the name). Diffusion models are very well suited for this task, because multiple transition states are often close to each other in energy even though they sometimes belong to different reaction classes, e.g. substitution vs. elimination. The diffusion model will start from a random guess, which it prints out and is very far away from any sensible chemical structure, and in each iteration improves the structures through denoising it. Think about it as the network actively diffusing the input towards the output (again hence the name). On top of that tsdiff does a good job of identifying reaction-patterns, by mapping the molecule onto a 2D graph.

Now we are testing it for a proton transfer between hydronium and water as well as a proton transfer between CH4 and CH3.

If you have a GPU I once again highly recommend using it, by setting the pytorch device.

In [1]:
# create train, validation and test set, but in this case
# we are only interested in testing custom cases, so only
# the test set will contain data.

from helpers.preprocessing_helper import preprocess_args, preprocess
from helpers.sampling_helper import sampling_args, sample

In [2]:
# class attributes contain parameters
prep_params = preprocess_args()
prep_params.train = 0.0
prep_params.valid = 0.0
# generate preprocessed files
preprocess(prep_params)

2it [00:00, 837.27it/s]


In [6]:
# run a pretrained model on the generated test data
sample_params = sampling_args()
#sample_params.device = "cuda"
# only use one model for faster calculation
# if you want to use all 8 available ones, just uncomment this line
sample_params.ckpt = [sample_params.ckpt[0]]
sample_params.n_steps = 5000
sample(sample_params)

[2026-05-08 13:39:50,581::test::INFO] <helpers.sampling_helper.sampling_args object at 0x7edb5115e6d0>
[2026-05-08 13:39:50,581::test::INFO] <helpers.sampling_helper.sampling_args object at 0x7edb5115e6d0>
[2026-05-08 13:39:50,583::test::INFO] Loading model...
[2026-05-08 13:39:50,583::test::INFO] Loading model...
[2026-05-08 13:39:50,613::test::INFO] load model from logs/trained_ckpt/ens0/checkpoints/best_ckpt.pt
[2026-05-08 13:39:50,613::test::INFO] load model from logs/trained_ckpt/ens0/checkpoints/best_ckpt.pt
[2026-05-08 13:39:50,668::test::INFO] Loading datasets...
[2026-05-08 13:39:50,668::test::INFO] Loading datasets...
[2026-05-08 13:39:50,670::test::INFO] Test file from data/scratch/random_split_42/test_data.pkl.
 Loading dataset...
[2026-05-08 13:39:50,670::test::INFO] Test file from data/scratch/random_split_42/test_data.pkl.
 Loading dataset...


0it [00:00, ?it/s]

Initial Position
tensor([[-1.1910e+01, -1.8440e+01, -1.0005e+01],
        [ 1.4768e+00,  8.7110e-01, -1.0608e+00],
        [-3.0840e-01, -1.9795e+01,  2.7137e+00],
        [-2.0377e+01, -1.7918e+01,  1.1829e+01],
        [-2.6667e+00, -1.1435e+01, -2.1607e+01],
        [-7.6162e+00, -1.3511e+01,  1.4469e+01],
        [ 1.6707e+01,  5.9802e+00,  4.3551e+00],
        [-2.0503e+00, -1.0440e+01, -1.2867e+01],
        [ 1.0295e+01,  1.8406e+01,  3.5174e-01],
        [-1.9278e+00,  2.2354e+01,  8.1388e+00],
        [-2.1630e+01,  2.4000e-02, -2.6832e+01],
        [ 8.3482e+00,  1.1577e+01, -3.0906e+00],
        [-8.9028e+00,  4.3331e+00, -1.2419e+01],
        [-1.7825e+01, -2.8411e+00, -5.3819e+00],
        [-1.0890e+01, -8.5693e+00,  2.2328e+00],
        [ 4.4067e+00, -1.3771e+01,  1.8083e+00]])


sample: 0it [00:00, ?it/s]

Generated Position
tensor([[-0.3192, -0.1920, -1.2261],
        [ 0.0808,  0.2305,  1.2386],
        [ 0.3298,  0.2233, -1.8440],
        [-0.6138, -1.1082, -1.4231],
        [-0.3337,  1.0021,  1.6612],
        [ 0.9171, -0.1429,  1.6195],
        [-0.0609, -0.0127, -0.0261],
        [ 0.7046,  0.7525,  0.9829],
        [ 1.3152,  0.1411,  1.5957],
        [-0.0659,  1.3425,  1.4319],
        [ 1.3323,  1.3842,  0.2665],
        [-0.0075, -0.0566,  0.0341],
        [-0.7088, -0.7680, -0.9431],
        [-1.5964, -1.1714, -0.4363],
        [-0.9449, -0.0613, -1.7116],
        [-0.0286, -1.5630, -1.2202]])


In [7]:
# visualize the generated transition state structures
import py3Dmol

def view_xyz(filename):
    with open(filename, "r") as f:
        xyz = f.read()

        viewer = py3Dmol.view(width=500, height=400)
        viewer.addModel(xyz, "xyz")

        # nice default style
        viewer.setStyle({
            "stick": {"radius": 0.15},
            "sphere": {"scale": 0.3}
        })

        viewer.zoomTo()
    return viewer

view_xyz("ts0.xyz")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [8]:
view_xyz("ts1.xyz")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Clustering and Training

A nice feature of the diffusion methodology is that when running this multiple times, different transition state structures will be found, if the PES is reasonably complex. If that is the case you can use tsdiffs clustering tool, which groups the reactions e.g. into competing reaction channels. Our test reactions are, however, too simple for that.

Regarding training there is, to the best of my knowledge, no short retraining opportunity for tsdiff, so if you want to add something you have to fully retrain from your seed.

If you are interested you can find the GitHub repository for tsdiff [here](https://github.com/seonghann/tsdiff).

## Where Do TS Predictors Work Well?

| System Type | Performance |
|------------|------------|
| Small organic reactions | Excellent |
| Proton transfer | Very strong |
| SN2 reactions | Strong |
| Rearrangements | Mixed |
| Large flexible systems | Challenging |
| Transition metals | Poor |
| Charged/solvent systems | Difficult |

Conclusion:
AI TS predictors are powerful but require hybrid workflows and validation.

# Now try tsdiff for your own reaction

Now for the fun part, where you create your own reaction and can use tsdiff to predict the transition state structure.

Therefore you will have to do the following:
- Create SMARTS for the reactions you are interested in, e.g. with ChatGPT
- Create xyz structure of reactants/products/whatever (this is just for workflow consistency with tsdiff)
- Create your new "test set" with the preprocessing routine
- Generate the transition state structure using the sampling routine

You can just append smarts.csv and structures.xyz in data/raw_data/ with the new smarts and xyz, but **beware of the atom ordering** in the smarts and the xyz! The atoms in the smarts need to have the same enumeration for reactants and products, and this ordering also needs to apply for the xyz coordinates. To give an example, if the reactant side of the smart has an O at position 2 and H at position three, but it's vice versa on the product side, this won't work.

In order to (only) calculate your new test reaction, you also need to adjust end_idx from 2 to 3.